# Medallion Pipeline — Visão Geral Executada

Este notebook percorre o pipeline de ponta a ponta: **Extract (API -> Bronze) -> Transform (MERGE -> Silver) -> Quality Gate -> Gold**.

Detalhes arquiteturais no `README.md` e em `docs/arquitetura.md`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.config import GOLD_PATH, get_spark, new_batch_id

spark = get_spark("pipeline-overview")
spark.sparkContext.setLogLevel("ERROR")
print(spark.version)

## 1. Extract — API -> Bronze (append imutável)

In [ ]:
from src.extract import extract_to_bronze

batch_id = new_batch_id()
for fonte in ["products", "carts", "users"]:
    n = extract_to_bronze(spark, fonte, batch_id)
    print(f"{fonte}: {n} registros no lote {batch_id[:8]}")

In [ ]:
bronze_products = spark.read.format("delta").load("../data/lakehouse/bronze/products")
bronze_products.printSchema()
bronze_products.select("record_pk", "batch_id", "ingested_at").show(3, truncate=False)

## 2. Transform — MERGE incremental Bronze -> Silver

Idempotência: a segunda execução não muda o resultado.

In [ ]:
from src.transform import transform_silver

print("--- Primeira execução (cria a Silver) ---")
print(transform_silver(spark, "products"))
print("--- Segunda execução (MERGE idempotente) ---")
print(transform_silver(spark, "products"))

In [ ]:
for fonte in ["carts", "users"]:
    print(transform_silver(spark, fonte))

In [ ]:
silver_products = spark.read.format("delta").load("../data/lakehouse/silver/products")
silver_products.select("id", "title", "category", "price", "rating", "updated_at").show(5, truncate=False)

## 3. Quality Gate — regras críticas antes da Gold

In [ ]:
import json
from src.quality import executar_quality_gate

report = executar_quality_gate(spark, incluir_gold=False)
print(json.dumps(report["resumo"], indent=2))

## 4. Gold — modelagem de consumo

In [ ]:
from src.gold import build_gold

contagens = build_gold(spark)
print(contagens)

In [ ]:
gold = "../data/lakehouse/gold"
spark.read.format("delta").load(f"{gold}/agg_kpis_gerais").show()
spark.read.format("delta").load(f"{gold}/agg_vendas_categoria").orderBy("receita", ascending=False).show(10, truncate=False)

In [ ]:
report_final = executar_quality_gate(spark, incluir_gold=True)
print(json.dumps(report_final["resumo"], indent=2))
spark.stop()

## Conclusões

1. **Bronze** preserva o histórico imutável por lote (batch_id + ingested_at) — auditoria e reprocessamento sem nova chamada à API.
2. **Silver** fica idempotente pelo MERGE por id + hash: reprocessar zero impacto (conferido na seção 2).
3. **Quality gate** bloqueia a publicação da Gold quando qualquer regra crítica falha — consumidores nunca veem dados não validados.
4. **Gold** entrega dimensão, fato e agregações prontas para Power BI / SQL Analytics.

**Próximos passos** (ver README): orquestração agendada, SCD2 no dim_produto, alerta sobre o quality report e deploy no Fabric/Databricks.